In [5]:
import pandas as pd
import re # será utilizado para desconsiderar letras maiúculas e minúsculas 
from unidecode import unidecode # utilizado para retirar acentos das palavras

# EXTRAIR

df = pd.read_csv("../dados/chamados_glpi_bruto.csv", sep=";")

In [6]:
# TRATAMENTO

# Excluir colunas
df = df.drop(columns=["Requerente - Requerente", "Requerente - Autor", "Atribuído - Técnico", "Categoria"])

# Exclusão de linha inteira com base em uma coluna
df = df[df["Entidade"].isin(["Entidade raizSaude","Entidade raizMonitoramento"])]
df = df[df["Status"] != "Fechado"]
df = df[df["Título"] == "CHAMADO MONITORAMENTO"]

#df["Descrição"] = df["Descrição"].str.replace(r'.*Equipamento', 'Equipamento', regex=True)
df = df.drop_duplicates(subset=["Descrição"], keep="last") # remover duplicadas

In [7]:
# Padronizar

# Substituição dos valores da coluna
df["Entidade"] = df["Entidade"].replace({
    "Entidade raizMonitoramento": "Monitoramento",
    "Entidade raizSaude": "Saúde"
})

In [8]:
# Criando atributos

def extrair_equipamento(descricao):

    if not isinstance(descricao, str):
        return "OUTROS"


    estruturado = re.search(r'5\)\s*Equipamento\s*:\s*([^\d\)]+?)(?:\d\))', unidecode(descricao), re.IGNORECASE) # pega o valor após "Equipamento"
    if estruturado and estruturado.group(1).strip(): # confere se há algum valor após "Equipamento"
        return estruturado.group(1).strip().upper() # retorna o valor

    relato = re.search(r'RELATE O PROBLEMA\s*:\s*(.+)', unidecode(descricao), re.IGNORECASE) # pega o valor após "RELATE O PROBLEMA"
    if relato:
        texto_relato = relato.group(1) # pega valor após "RELATE O PROBLEMA"

        equipamentos = {
            "CAMERA": ["CAMERA", "CAMERAS", "CAM", "CAMS"],
            "DVR": ["DVR"],
            "NVR": ["NVR"],
            "SPEED": ["SPEED", "SPEEDS"],
            "ALARME": ["ALARMES", "ALARME"]
        }

        for chave, valor in equipamentos.items():
            for varicao in valor:
                if varicao in texto_relato.upper():
                    return chave

    return "OUTROS"
    

df["Equipamento"] = df["Descrição"].apply(extrair_equipamento) # aplica a função em cada linha de "Descrição"


In [9]:
def extrair_unidade(unidade):

    if not isinstance(unidade,  str):
        return "NAO IDENTIFICADO"

    tipos = {
        "UBS": ["UBS"],
        "ESCOLA": ["ESCOLA"],
        "SAAE": ["SAAE"],
        "CMEI": ["CMEI"],
        "AMBULATORIO": ["AMBULATORIO"],
        "GINASIO": ["GINASIO"],
        "CEMITERIO": ["CEMITERIO"],
        "GARAGEM": ["GARAGEM"]
    }

    for chave, valor in tipos.items():
        for variacao in valor:
            if variacao in unidecode(unidade).upper():
                return chave


    return "NAO IDENTIFICADO"

df["Unidade"] = df["Descrição"].apply(extrair_unidade)


In [10]:
setores = {
    "UBS": "SAUDE",
    "AMBULATORIO": "SAUDE",
    "ESCOLA": "EDUCACAO",
    "CMEI": "EDUCACAO",
    "SAAE": "SANEAMENTO",
    "GINASIO": "ESPORTE E LAZER",
    "CEMITERIO": "SERVICOS URBANOS",
    "GARAGEM": "SERVICOS URBANOS",
    "NAO IDENTIFICADO": "NAO IDENTIFICADO"
}

df["Setor"] = df["Unidade"].map(setores)

In [11]:
df["Data de abertura"]= pd.to_datetime(df["Data de abertura"], format="%d/%m/%Y %H:%M")
df["Última atualização"]= pd.to_datetime(df["Última atualização"], format="%d/%m/%Y %H:%M")

def calcular_tempo(base):
    
    if base["Status"] == "Solucionado":   
        return str(base["Última atualização"] - base["Data de abertura"])
    else:
        return "NAO SOLUCIONADO"

df["Tempo para solução"] = df.apply(calcular_tempo, axis=1) # "axis=1" significa que irá percorrer na horizontal

# Remover coluna descrição após utilização
df = df.drop(columns=["Descrição"])


In [12]:
# Salvar novo arquivo

df_tratado = df.to_csv("../dados/chamados_glpi_tratados.csv", index=False, sep=";", encoding="utf-8-sig")

In [13]:
# Leitura

df.head(20)

,ID,Título,Entidade,Status,Última atualização,Data de abertura,Prioridade,Tempo para solução,Equipamento,Unidade,Setor
1,11 946,CHAMADO MONITORAMENTO,Monitoramento,Solucionado,2026-09-10 07:46:00,2026-09-09 06:11:00,Média,1 days 01:35:00,SPEED,UBS,SAUDE
2,11 940,CHAMADO MONITORAMENTO,Saúde,Novo,2026-09-10 07:44:00,2026-09-08 11:48:00,Média,NAO SOLUCIONADO,CAMERA,AMBULATORIO,SAUDE
3,11 955,CHAMADO MONITORAMENTO,Monitoramento,Solucionado,2026-09-10 07:44:00,2026-09-09 12:38:00,Média,0 days 19:06:00,CAMERA,ESCOLA,EDUCACAO
4,11 952,CHAMADO MONITORAMENTO,Monitoramento,Solucionado,2026-09-10 07:43:00,2026-09-09 09:35:00,Alta,0 days 22:08:00,NVR,ESCOLA,EDUCACAO
5,11 951,CHAMADO MONITORAMENTO,Monitoramento,Solucionado,2026-09-10 07:43:00,2026-09-09 08:36:00,Alta,0 days 23:07:00,DVR,UBS,SAUDE
6,11 950,CHAMADO MONITORAMENTO,Monitoramento,Solucionado,2026-09-10 07:41:00,2026-09-09 08:35:00,Alta,0 days 23:06:00,DVR,UBS,SAUDE
7,11 949,CHAMADO MONITORAMENTO,Monitoramento,Solucionado,2026-09-10 07:40:00,2026-09-09 08:33:00,Alta,0 days 23:07:00,NVR,SAAE,SANEAMENTO
8,11 948,CHAMADO MONITORAMENTO,Monitoramento,Solucionado,2026-09-10 07:40:00,2026-09-09 08:28:00,Alta,0 days 23:12:00,DVR,ESCOLA,EDUCACAO
9,11 947,CHAMADO MONITORAMENTO,Monitoramento,Solucionado,2026-09-10 07:39:00,2026-09-09 08:18:00,Alta,0 days 23:21:00,DVRNVRCAMERAALARME,SAAE,SANEAMENTO
10,11 944,CHAMADO MONITORAMENTO,Monitoramento,Solucionado,2026-09-10 07:36:00,2026-09-08 23:46:00,Média,1 days 07:50:00,OUTROS,UBS,SAUDE
